# Test-environment report

Frozen Phase-1 checkpoint + trained change-point detector, driven through scripted
meta-test episodes. Two results:

1. **Detector performance** -- threshold sweep (TPR / FPR / F1 / detection delay) over
   *change* and *no-change* episodes, plus the as-deployed operating point.
2. **Recovery time** -- team-return dip after the switch, with the detector reset vs. a
   *passive* runner (reset disabled = the `- change-point detector` ablation).

Point `P1_CKPT` / `DETECTOR` at your run, then *Run All*.

In [ ]:
import sys, json, time
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if (ROOT / "method").exists(): sys.path.insert(0, str(ROOT))

from utils.scenario import Scenario
from method.io import load_phase1, load_detector
from method.training.meta_test import MetaTestRunner, MetaTestConfig, RegimeChangeEvent
from method.training.regime import Regime
from scripts.recovery_eval import (run_episode, trailing_mean, recovery_metrics,
                                   deployed_point, threshold_sweep)

# ---------- what is under test ----------
RUN        = ROOT / "runs" / "full_20260908_223140"          # <-- edit
P1_CKPT    = sorted(RUN.glob("phase1_checkpoint_*.pt"))[-1]
DETECTOR   = RUN / "phase2_detector.pt"
CFG        = json.loads((P1_CKPT.parent / "args.json").read_text()).get("config_file", "world_config_5v.yaml")

REPEATS      = 15
HORIZON      = 140
CHANGE_STEP  = 40
TARGET_AGENT = 0
SMOOTH_W     = 12
REC_BAND     = 0.10
NOMINAL, SHIFTED = Regime(1.10, 0.16), Regime(1.52, 0.36)
print("ckpt:", P1_CKPT.name, "| detector:", DETECTOR.name, "| config:", CFG)

In [ ]:
trainer  = load_phase1(P1_CKPT, scenario_factory=lambda: Scenario(config_file=CFG), config=None)
detector = load_detector(DETECTOR, phase1_trainer=trainer, config=None).eval()
NA = trainer.n_agents
factory  = lambda: Scenario(config_file=CFG)
run_with = MetaTestRunner(trainer, detector, factory, MetaTestConfig())
run_pass = MetaTestRunner(trainer, detector, factory,
                          MetaTestConfig(threshold_C=1.01, trigger_persistence=10**9))
print(f"n_agents={NA}  detector window={detector.window}  raw_in={detector.input_proj.in_features}")

In [ ]:
nominal_all = [RegimeChangeEvent(0, i, NOMINAL) for i in range(NA)]
change_ev   = nominal_all + [RegimeChangeEvent(CHANGE_STEP, TARGET_AGENT, SHIFTED)]

t0 = time.time(); ch_with, ch_pass, nochange = [], [], []
for r in range(REPEATS):
    torch.manual_seed(1000+r); ch_with.append(run_episode(run_with, HORIZON, change_ev))
    torch.manual_seed(1000+r); ch_pass.append(run_episode(run_pass, HORIZON, change_ev))
    torch.manual_seed(5000+r); nochange.append(run_episode(run_with, HORIZON, nominal_all))
    if r % 5 == 0: print(f"  [{time.time()-t0:4.0f}s] {r+1}/{REPEATS}")
print("done", round(time.time()-t0), "s")

## 1. Detector performance

In [ ]:
p_change = np.stack([e["p"][:, TARGET_AGENT] for e in ch_with])
p_noch   = np.stack([e["p"][:, TARGET_AGENT] for e in nochange])
thrs  = np.round(np.arange(0.10, 0.96, 0.05), 2)
sweep = threshold_sweep(p_change, [CHANGE_STEP]*len(ch_with), p_noch, HORIZON, thrs)
sw = pd.DataFrame(sweep)[["thr","tpr","fpr","f1","delay","TP","FP","FN","TN"]]
best = sw.loc[sw["f1"].idxmax()]
deployed = deployed_point(ch_with, nochange, TARGET_AGENT, CHANGE_STEP, HORIZON)
display(sw.style.hide(axis="index").format(precision=3))
print(f"best-F1 @ thr {best.thr:.2f}:  F1={best.f1:.3f}  TPR={best.tpr:.2f}  FPR={best.fpr:.2f}  delay={best.delay:.1f}")
print(f"as-deployed  (threshold_C={run_with.cfg.threshold_C}, persistence={run_with.cfg.trigger_persistence}):  "
      f"F1={deployed['f1']:.3f}  TPR={deployed['tpr']:.2f}  FPR={deployed['fpr']:.2f}  delay={deployed['delay']:.1f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(sw.thr, sw.tpr, "s-", label="TPR (detection)")
ax[0].plot(sw.thr, sw.fpr, "^-", label="FPR (false alarm)")
ax[0].plot(sw.thr, sw.f1,  "o-", label="F1")
ax[0].axvline(best.thr, color="k", ls="--", lw=1); ax[0].set_xlabel("threshold"); ax[0].legend(); ax[0].set_title("Detector vs threshold")
ax[1].plot(sw.thr, sw.delay, "d-", color="C3"); ax[1].axvline(best.thr, color="k", ls="--", lw=1)
ax[1].set_xlabel("threshold"); ax[1].set_ylabel("mean detection delay (steps)"); ax[1].set_title("Detection delay")
plt.tight_layout(); plt.show()

In [ ]:
# saw-tooth: p-trace, true change, as-deployed reset
fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
for ax, e in zip(axes, ch_with[:3]):
    ax.plot(e["p"][:, TARGET_AGENT], color="C0", lw=1.4, label="$p_t$")
    ax.axhline(run_with.cfg.threshold_C, ls="--", color="k", lw=.7)
    ax.axvline(CHANGE_STEP, color="C3", lw=2, label=f"true change t={CHANGE_STEP}")
    fr = np.where(e["reset"][:, TARGET_AGENT])[0]
    if len(fr): ax.axvline(int(fr[0]), color="C2", ls=":", lw=2, label=f"reset t={int(fr[0])}")
    ax.set_ylim(-.05, 1.05); ax.set_ylabel("$p_t$")
axes[0].legend(fontsize=8, loc="upper left"); axes[-1].set_xlabel("step")
fig.suptitle("Change point vs detection (as deployed)"); plt.tight_layout(); plt.show()

## 2. Recovery time  (reset vs. passive re-adaptation)

In [ ]:
def agg(eps):
    m = [recovery_metrics(e["team_r"], CHANGE_STEP, SMOOTH_W, REC_BAND) for e in eps]
    rec = [x for x in m if x["recovered"]]
    return dict(recovery_time_mean=np.mean([x["recovery_time"] for x in m]),
                recovery_time_median=np.median([x["recovery_time"] for x in m]),
                recovery_time_if_recovered=np.mean([x["recovery_time"] for x in rec]) if rec else np.nan,
                non_recovery_rate=1 - len(rec)/len(m),
                transient_cost=np.mean([x["transient_cost"] for x in m]),
                pre_switch_baseline=np.mean([x["baseline"] for x in m]))
tbl = pd.DataFrame({"with detector": agg(ch_with), "passive": agg(ch_pass)})
reexpl = np.mean([np.sum(e["mode"][CHANGE_STEP:, TARGET_AGENT] == "explore") for e in ch_with])
tbl.loc["re_exploration_steps"] = [reexpl, np.nan]
display(tbl.style.format(precision=2))
d = tbl.loc["recovery_time_mean", "passive"] - tbl.loc["recovery_time_mean", "with detector"]
print(f"recovery-time reduction (passive - with): {d:+.2f} steps "
      f"({100*d/max(tbl.loc['recovery_time_mean','passive'],1e-6):+.0f}%)")

In [ ]:
sm_with = np.stack([trailing_mean(e["team_r"], SMOOTH_W) for e in ch_with])
sm_pass = np.stack([trailing_mean(e["team_r"], SMOOTH_W) for e in ch_pass])
x = np.arange(HORIZON)
plt.figure(figsize=(11, 4.4))
for sm, c, lab in [(sm_with, "C0", "with detector"), (sm_pass, "C1", "passive")]:
    mu, sd = sm.mean(0), sm.std(0)
    plt.plot(x, mu, color=c, lw=2, label=lab); plt.fill_between(x, mu-sd, mu+sd, color=c, alpha=.15)
plt.axvline(CHANGE_STEP, color="C3", lw=2, label=f"switch t={CHANGE_STEP}")
plt.axhline(float(np.mean([recovery_metrics(e['team_r'], CHANGE_STEP, SMOOTH_W, REC_BAND)['baseline'] for e in ch_with])),
            color="k", ls="--", lw=.8, label="pre-switch baseline")
plt.xlabel("step"); plt.ylabel(f"team return (trailing mean, w={SMOOTH_W})")
plt.title("Recovery after the switch"); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

## 3. Example rescue episode

In [ ]:
torch.manual_seed(2024)
ep = run_episode(run_with, HORIZON, change_ev)
health = []; runner = run_with
# re-run one episode capturing victim health each step
runner.reset_state(); runner._regime_events.clear(); runner._attack_events.clear(); runner.schedule(change_ev)
hp, pr, rew = [], [], []
for _ in range(HORIZON):
    s = runner.step(render=False)
    surv = runner.env.scenario._survivals
    hp.append(float(min(v.health[0].item() for v in surv)))
    pr.append(s.detector_p[TARGET_AGENT]); rew.append(float(np.sum(s.rewards)))
resc = int(sum(bool(v.rescued[0]) for v in runner.env.scenario._survivals))
fig, ax = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
ax[0].plot(hp, color="C3"); ax[0].axvline(CHANGE_STEP, color="C3", lw=2)
ax[0].set_ylabel("min victim health"); ax[0].set_title(f"{resc}/{len(runner.env.scenario._survivals)} rescued")
ax[1].plot(pr, color="C0"); ax[1].axhline(run_with.cfg.threshold_C, ls="--", color="k", lw=.7)
ax[1].axvline(CHANGE_STEP, color="C3", lw=2); ax[1].set_ylabel("detector $p_t$"); ax[1].set_ylim(-.05, 1.05)
ax[2].plot(np.cumsum(rew), color="C2"); ax[2].axvline(CHANGE_STEP, color="C3", lw=2)
ax[2].set_ylabel("cumulative return"); ax[2].set_xlabel("step")
plt.tight_layout(); plt.show()